#### Relevant Imports

In [8]:
from sqlalchemy import create_engine, text

**Engine**  
Engine is the abstract object from SQLAlchmey which connects to a host to particular Database  
In an an engine further connections can be created to query data  
This generic concepts make it Database agnostic

In [9]:
# There is an engine instance created, which can handle multiple connetions
sql_engine = create_engine("mysql+pymysql://rfamro:@mysql-rfam-public.ebi.ac.uk:4497/Rfam")

**Connection and Query**
Connection object creates a connection (its within the object)  
Then it is used to raise query (Execute Query String)

In [10]:
# Create a Connection in engine and raise a query to get data
conn_1 = sql_engine.connect ()
result = conn_1.execute(text("SELECT COUNT(*) FROM full_region;"))

# Fetch the result content
for row in result:

    print (row)
    print (type(row))



(10645620,)
<class 'sqlalchemy.engine.row.Row'>


In [11]:
# Get more data, parallelly from another connection in same engine
conn_2 = sql_engine.connect ()
result = conn_2.execute(text("SELECT * FROM full_region LIMIT 10;"))

# Fetch the result content
print (result.keys())
for row in result:

    print (row)
    print (type(row))


Exception during reset or similar
Traceback (most recent call last):
  File "/Users/rajuboddula/Downloads/Gen AI - Outskill/Projects/GenAIEngineering-Cohort1/Week6/.venv/lib/python3.12/site-packages/pymysql/connections.py", line 834, in _write_bytes
    self._sock.sendall(data)
  File "/Users/rajuboddula/.local/share/uv/python/cpython-3.12.4-macos-aarch64-none/lib/python3.12/ssl.py", line 1211, in sendall
    v = self.send(byte_view[count:])
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/rajuboddula/.local/share/uv/python/cpython-3.12.4-macos-aarch64-none/lib/python3.12/ssl.py", line 1180, in send
    return self._sslobj.write(data)
           ^^^^^^^^^^^^^^^^^^^^^^^^
ssl.SSLEOFError: EOF occurred in violation of protocol (_ssl.c:2406)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/rajuboddula/Downloads/Gen AI - Outskill/Projects/GenAIEngineering-Cohort1/Week6/.venv/lib/python3.12/site-packages/sqlalchemy/poo

RMKeyView(['rfam_acc', 'rfamseq_acc', 'seq_start', 'seq_end', 'bit_score', 'evalue_score', 'cm_start', 'cm_end', 'truncated', 'type', 'is_significant'])
('RF00061', 'AF009606.1', 2, 354, 434.2, '3.8e-132', 1, 352, '0', 'full', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'AF029248.1', 20265, 20371, 128.5, '7.6e-24', 1, 107, '0', 'full', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'AF201929.1', 20103, 20209, 128.5, '7.6e-24', 1, 107, '0', 'full', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'AF207902.1', 20103, 20209, 128.5, '7.6e-24', 1, 107, '0', 'full', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'JN874562.1', 20063, 20172, 123.1, '2e-22', 1, 107, '0', 'seed', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'FJ425184.1', 19998, 20104, 122.0, '3.9e-22', 1, 107, '0', 'full', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'FJ425187.1', 20000, 20106, 122.0, '3.9e-22', 1, 107, '0', 'full', 1)
<class 'sqlalchemy.engine.row.Row'>
('RF00182', 'DQ339101.1', 20

In [12]:
# For a Given value of rfam_acc, how many Ncbi Id
result = conn_1.execute (text(
                             "SELECT COUNT(DISTINCT fn.ncbi_id) AS ncbi_count\
                            FROM family_ncbi fn\
                            WHERE fn.rfam_acc = 'RF01530';"))

for row in result:

    print (row)    


(14,)


**Data from multiple tables**  
Data from multiple table can be collated by the linking fields by JOIN queries

In [13]:
# For a Given value of rfam_acc, how many species are present in another table based on its ncbi_id
result = conn_1.execute (text(
                             "SELECT DISTINCT fn.rfam_acc, fn.ncbi_id, t.species AS species\
                            FROM family_ncbi fn\
                            JOIN taxonomy t ON fn.ncbi_id = t.ncbi_id\
                            WHERE fn.rfam_acc = 'RF01530'"))

for row in result:

    print (row)    


('RF01530', 190650, 'Caulobacter crescentus CB15')
('RF01530', 565050, 'Caulobacter crescentus NA1000')
('RF01530', 1736578, 'Caulobacter sp. Root655')
('RF01530', 2172650, 'Caulobacter radicis')
('RF01530', 1813876, 'Phenylobacterium hankyongense')
('RF01530', 1445034, 'Phenylobacterium kunshanense')
('RF01530', 69395, 'Caulobacter henricii')
('RF01530', 2170551, 'Phenylobacterium soli')
('RF01530', 2015570, 'Alphaproteobacteria bacterium PA2')
('RF01530', 2803784, 'Phenylobacterium glaciei')
('RF01530', 1736442, 'Phenylobacterium sp. Root1277')
('RF01530', 1914756, 'Phenylobacterium deserti')
('RF01530', 450851, 'Phenylobacterium zucineum HLK1')
('RF01530', 69666, 'Caulobacter sp. FWC38')


In [14]:
# Consolidated information from 3 tables
result = conn_1.execute (text(
                                "SELECT fn.rfam_id, fn.ncbi_id, t.species, f.rfam_id AS family_rfam_id, f.auto_wiki, f.description\
                                FROM family_ncbi fn\
                                JOIN taxonomy t ON fn.ncbi_id = t.ncbi_id\
                                JOIN family f ON fn.rfam_acc = f.rfam_acc\
                                WHERE fn.rfam_acc = 'RF01530'"))

rows = result.fetchall ()
print (rows)
  

[('CC3664', 190650, 'Caulobacter crescentus CB15', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 565050, 'Caulobacter crescentus NA1000', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 1736578, 'Caulobacter sp. Root655', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 2172650, 'Caulobacter radicis', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 1813876, 'Phenylobacterium hankyongense', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 1445034, 'Phenylobacterium kunshanense', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 69395, 'Caulobacter henricii', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 2170551, 'Phenylobacterium soli', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 2015570, 'Alphaproteobacteria bacterium PA2', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 2803784, 'Phenylobacterium glaciei', 'CC3664', 2085, 'Caulobacter sRNA CC3664'), ('CC3664', 1736442, 'Phenylobacterium sp. Root1277', 'CC3664', 2085, '